# Train
Run all setup cells first (Colab clone, pip install, imports, definitions, device setup).
Then **Step 1:** smoke-test the pipeline with a tiny run. **Step 2:** launch full training.

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('Redshift-Aware-Spectral-Transformer'):
        !git clone https://github.com/Guillermo-03/Redshift-Aware-Spectral-Transformer.git
    %cd Redshift-Aware-Spectral-Transformer/src

In [ ]:
%pip install -q torch datasets tqdm numpy h5py astropy

In [ ]:
import os, sys, math
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import config

In [ ]:
%run data.ipynb
%run model.ipynb

In [ ]:
def setup_environment():
    """Mount Google Drive when running in Colab; fall back gracefully on auth failure.

    Returns:
        tuple: (checkpoint_dir, cache_path)
            checkpoint_dir (str): Absolute path to the checkpoint directory.
            cache_path (str or None): Path for the clean dataset cache, or None if
                Drive is unavailable (cache disabled, data reloaded from HuggingFace).
    """
    in_colab = 'google.colab' in sys.modules
    if in_colab:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            drive_root = '/content/drive/MyDrive/RAST'
            checkpoint_dir = f'{drive_root}/checkpoints'
            cache_path = f'{drive_root}/desi_clean_200k'
            print('Running in Colab — checkpoints and cache saving to Google Drive.')
        except Exception as e:
            print(f'Warning: Drive mount failed ({e}).')
            print('Falling back to /content/checkpoints — checkpoints will be lost on disconnect.')
            checkpoint_dir = '/content/checkpoints'
            cache_path = None
    else:
        checkpoint_dir = os.path.abspath(os.path.join(os.getcwd(), '..', config.CHECKPOINT_DIR))
        cache_path = None
        print('Running locally — checkpoints saving to checkpoints/.')
    os.makedirs(checkpoint_dir, exist_ok=True)
    return checkpoint_dir, cache_path

In [ ]:
def get_scheduler(optimizer):
    """Build a scheduler with linear warmup followed by cosine annealing.

    Args:
        optimizer: AdamW optimizer instance.

    Returns:
        torch.optim.lr_scheduler.LambdaLR
    """
    def lr_lambda(epoch):
        if epoch < config.WARMUP_EPOCHS:
            return (epoch + 1) / config.WARMUP_EPOCHS
        progress = (epoch - config.WARMUP_EPOCHS) / max(1, config.EPOCHS - config.WARMUP_EPOCHS)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
def joint_loss(recon_pred, recon_target, z_pred, z_target, lambda_z=config.LAMBDA_REDSHIFT):
    """Compute the joint reconstruction and redshift prediction loss.

    Args:
        recon_pred (torch.Tensor): Predicted patches at masked positions, shape (B, n_masked, PATCH_SIZE).
        recon_target (torch.Tensor): Ground-truth patches at masked positions, shape (B, n_masked, PATCH_SIZE).
        z_pred (torch.Tensor): Predicted redshift values, shape (B, 1).
        z_target (torch.Tensor): Ground-truth redshift values, shape (B, 1).
        lambda_z (float): Weighting factor for the redshift loss term.

    Returns:
        tuple: (total_loss, L_recon, lambda_z * L_z) — all scalar tensors.
    """
    L_recon = F.mse_loss(recon_pred, recon_target)
    L_z_weighted = lambda_z * F.mse_loss(z_pred, z_target)
    return L_recon + L_z_weighted, L_recon, L_z_weighted

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """Run one full pass over the training set.

    Args:
        model (SpectralTransformer): The model being trained.
        loader (DataLoader): Training data loader.
        optimizer: Optimizer instance.
        device (torch.device): Target compute device.

    Returns:
        tuple: (mean_total_loss, mean_recon_loss, mean_z_loss) over the epoch.
    """
    model.train()
    total, recon_sum, z_sum = 0.0, 0.0, 0.0

    for flux, redshift in tqdm(loader, desc='Train', leave=False):
        flux     = flux.to(device)
        redshift = redshift.unsqueeze(1).to(device)
        B = flux.shape[0]

        mask = random_mask(B, config.N_PATCHES).to(device)
        patches = flux.reshape(B, config.N_PATCHES, config.PATCH_SIZE)
        n_masked = mask.sum(dim=1)[0].item()
        target_patches = patches[mask].reshape(B, int(n_masked), config.PATCH_SIZE)

        recon_pred, z_pred = model(flux, mask)
        loss, L_recon, L_z = joint_loss(recon_pred, target_patches, z_pred, redshift)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total     += loss.item()
        recon_sum += L_recon.item()
        z_sum     += L_z.item()

    n = len(loader)
    return total / n, recon_sum / n, z_sum / n


def validate(model, loader, device):
    """Evaluate the model on the validation set without gradient updates.

    Args:
        model (SpectralTransformer): The model to evaluate.
        loader (DataLoader): Validation data loader.
        device (torch.device): Target compute device.

    Returns:
        tuple: (mean_total_loss, mean_recon_loss, mean_z_loss) over the epoch.
    """
    model.eval()
    total, recon_sum, z_sum = 0.0, 0.0, 0.0

    with torch.no_grad():
        for flux, redshift in tqdm(loader, desc='Val', leave=False):
            flux     = flux.to(device)
            redshift = redshift.unsqueeze(1).to(device)
            B = flux.shape[0]

            mask = random_mask(B, config.N_PATCHES).to(device)
            patches = flux.reshape(B, config.N_PATCHES, config.PATCH_SIZE)
            n_masked = mask.sum(dim=1)[0].item()
            target_patches = patches[mask].reshape(B, int(n_masked), config.PATCH_SIZE)

            recon_pred, z_pred = model(flux, mask)
            loss, L_recon, L_z = joint_loss(recon_pred, target_patches, z_pred, redshift)

            total     += loss.item()
            recon_sum += L_recon.item()
            z_sum     += L_z.item()

    n = len(loader)
    return total / n, recon_sum / n, z_sum / n

In [ ]:
checkpoint_dir, cache_path = setup_environment()
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {device}')

## Step 1 — Tiny training run
Confirm the full pipeline works before committing to full training.
Loss should decrease and neither recon nor z*5 should be NaN.

In [ ]:
torch.manual_seed(42)

tiny_train, tiny_val = get_dataloaders(batch_size=4, n_train=16, n_val=8, chunk_size=1000)
tiny_model = SpectralTransformer().to(device)
tiny_opt   = AdamW(tiny_model.parameters(), lr=config.LR, weight_decay=1e-4)

for epoch in range(3):
    tl, tr, tz = train_one_epoch(tiny_model, tiny_train, tiny_opt, device)
    vl, vr, vz = validate(tiny_model, tiny_val, device)
    print(
        f'Epoch {epoch+1}/3 | '
        f'train  total={tl:.4f}  recon={tr:.4f}  z*5={tz:.4f} | '
        f'val    total={vl:.4f}  recon={vr:.4f}  z*5={vz:.4f}'
    )

print('Pipeline check passed — proceed to full training.')

## Step 2 — Full Training

Only run this after Step 1 confirms the pipeline is working end-to-end.

**What this run does:**
- Loads exactly **180K train / 20K val** ZWARN=0 DESI spectra by scanning up to 1M raw examples in 50K chunks
- Trains a **~35M parameter** SpectralTransformer for **50 epochs**
- 5-epoch linear warmup → cosine annealing down to 0
- Best checkpoint saved automatically whenever validation loss improves


In [ ]:
torch.manual_seed(42)

train_loader, val_loader = get_dataloaders(
    n_train=180_000,
    n_val=20_000,
    chunk_size=50_000,
    max_raw=1_000_000,
    cache_path=cache_path,
)
model = SpectralTransformer().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer = AdamW(model.parameters(), lr=config.LR, weight_decay=1e-4)
scheduler = get_scheduler(optimizer)
best_val_loss = float('inf')
start_epoch = 0

# Resume from latest checkpoint if available
latest_path = os.path.join(checkpoint_dir, 'latest.pt')
if os.path.exists(latest_path):
    ckpt = torch.load(latest_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    best_val_loss = ckpt['best_val_loss']
    start_epoch = ckpt['epoch'] + 1
    print(f'Resuming — next epoch: {start_epoch + 1}/{config.EPOCHS}  best val loss so far: {best_val_loss:.4f}')
else:
    print('Starting training from scratch.')

for epoch in range(start_epoch, config.EPOCHS):
    tl, tr, tz = train_one_epoch(model, train_loader, optimizer, device)
    vl, vr, vz = validate(model, val_loader, device)
    scheduler.step()

    lr_now = scheduler.get_last_lr()[0]
    print(
        f'Epoch {epoch+1:3d}/{config.EPOCHS} | '
        f'train  total={tl:.4f}  recon={tr:.4f}  z*5={tz:.4f} | '
        f'val    total={vl:.4f}  recon={vr:.4f}  z*5={vz:.4f} | '
        f'lr={lr_now:.2e}'
    )

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best.pt'))
        print(f'  -> Saved best checkpoint')

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
    }, latest_path)